# 🔍 Vulnerability Lookup Tool
This notebook demonstrates how to authenticate with the Checkmarx API and search for known vulnerabilities in software packages. It includes:
- API authentication
- Package ID lookup
- Vulnerability extraction
- Export to CSV


### Imports

In [8]:
import requests
import json
import uuid
import datetime
import os
import time
from collections import defaultdict
import csv

## 1. Authentication Setup

#### Update the following Variables:
- `TENANT_NAME`: The name of your CxOne tenant
- `TENANT_REGION`: The geo region of your tenant. ie. anz, sng, deu, etc..
- `REFRESH_TOKEN`: Your API Key from the CxOne Platform or generated Refresh Token


In [ ]:

#UPDATE WITH YOUR VALUES
TENANT_NAME = '<TENANT_NAME>'
REFRESH_TOKEN = '<API_KEY>'  # Replace with your actual refresh token
TENANT_REGION = '<TENANT_REGION>'  # e.g., 'anz', 'sng', etc.

In [ ]:

# Validate that no values were left blank
assert all([TENANT_NAME, REFRESH_TOKEN, TENANT_REGION]), "Please fill in all configuration values above."

AUTH_URL = f'https://{TENANT_REGION}.iam.checkmarx.net/auth/realms/{TENANT_NAME}/protocol/openid-connect/token'

payload = {
    'grant_type': 'refresh_token',
    'client_id': 'ast-app',
    'refresh_token': REFRESH_TOKEN
}

try:
    response = requests.post(AUTH_URL, data=payload)
    response.raise_for_status()

    token_data = response.json()
    access_token = token_data.get('access_token')
    new_refresh_token = token_data.get('refresh_token')

    if access_token:
        print("Access token:", access_token)
        if new_refresh_token:
            print("New refresh token:", new_refresh_token)
    else:
        print("No access token found:", token_data)

except requests.exceptions.HTTPError as http_err:
    print(f"HTTP error occurred: {http_err}")
    print("Response body:", response.text)
except Exception as err:
    print(f"Other error occurred: {err}")


## 2. Define Target Packages

Update `package_manager` and `package_name` to your desired input


`package_manager` must be one of the following: Dart, Go, Ios, Maven, Npm, Nuget, Perl, Php, Python, Ruby

In [ ]:
#UPDATE WITH YOUR VALUES
package_manager = "Maven"  #One of: Dart, Go, Ios, Maven, Npm, Nuget, Perl, Php, Python, Ruby
package_name = "org.springframework.data:spring-data-rest-webmvc"  # Example: "express", "spring-boot", etc

In [ ]:

ACCESS_TOKEN = access_token

url = f"https://{TENANT_REGION}.ast.checkmarx.net/api/sca/packages/{package_manager}/{package_name}/versions" 

headers = {
    'Authorization': f'Bearer {ACCESS_TOKEN}',
    'Content-Type': 'application/json',
    'Accept': 'application/json'
}
# Query parameters
params = {
    "OrderBy": "version"
}

# Make the GET request
response = requests.get(url, headers=headers, params=params)

# Check and print the response
if response.status_code == 200:
    data = response.json()
    legacy_ids = [item.get("legacyPackageId") for item in data if "legacyPackageId" in item]
    print(legacy_ids)
else:
    print(f"Request failed with status code {response.status_code}")
    print(response.text)


## 3. Search for Vulnerabilities & Export to CSV


Uncomment `payload` array and input specific package name and version if desired

In [ ]:
ACCESS_TOKEN = access_token

# POST URL
url = f'https://{TENANT_REGION}.ast.checkmarx.net/api/sca/vulnerabilities/search-requests'

# Headers
headers = {
    'Authorization': f'Bearer {ACCESS_TOKEN}',
    'Content-Type': 'application/json',
    'Accept': 'application/json'
}

# Uncomment if you want to use a specific package name and version
# payload = ["Python-Pillow-1.1"]

# Example payload of all versions of a package from above
payload = legacy_ids

def write_vulnerabilities_to_csv(package_vulns, filename):
    with open(filename, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow([
            "Package", "CVE ID", "Severity", "Published Date",
            "Description", "EPSS Score", "EPSS Percentile"
        ])

        for package, vulns in package_vulns.items():
            for vuln in vulns:
                cve_id = vuln.get('id', 'N/A')
                description = vuln.get('description', 'No description provided.').replace('\n', ' ').strip()
                severity = vuln.get('severity', 'Unknown')
                publish_date = vuln.get('publishDate', 'Unknown')

                epss_data = vuln.get('epssData', {}) or {}
                epss_score = epss_data.get('epss', 'N/A')
                epss_percentile = epss_data.get('percentile', 'N/A')

                writer.writerow([
                    package, cve_id, severity, publish_date,
                    description, epss_score, epss_percentile
                ])

try:
    # Make the POST request
    response = requests.post(url, headers=headers, json=payload)
    response.raise_for_status()

    # Parse the JSON response
    vulnerabilities = response.json()

    # Organize vulnerabilities by package
    package_vulns = defaultdict(list)

    for vuln in vulnerabilities:
        for package_id in vuln.get('packageIds', []):
            package_vulns[package_id].append(vuln)

    # Output results formatted by package
    for package, vulns in package_vulns.items():
        print(f"\nPackage: {package}")
        if not vulns:
            print("  No vulnerabilities found.")
        else:
            for vuln in vulns:
                cve_id = vuln.get('id', 'N/A')
                description = vuln.get('description', 'No description provided.')
                severity = vuln.get('severity', 'Unknown')
                publish_date = vuln.get('publishDate', 'Unknown')

                # Safe access to nested dicts to avoid NoneType errors
                epss_data = vuln.get('epssData', {}) or {}
                epss_score = epss_data.get('epss', 'N/A')
                epss_percentile = epss_data.get('percentile', 'N/A')

                print(f"  CVE ID: {cve_id}")
                print(f"    Severity: {severity}")
                print(f"    Published: {publish_date}")
                print(f"    Description: {description}")
                print(f"    EPSS Score: {epss_score} (Percentile: {epss_percentile})")
    # Write to CSV
    write_vulnerabilities_to_csv(package_vulns, "vulnerabilities_output.csv")
    print("\n Vulnerabilities exported to 'vulnerabilities_output.csv'")
    
except requests.exceptions.HTTPError as http_err:
    print(f"HTTP error occurred: {http_err}")
    print("Response body:", response.text)
except Exception as err:
    print(f"Other error occurred: {err}")

